# MiniGPT From Scratch — AI Engineering Assignment
**Purpose:** Train a small causal Transformer on Tiny Shakespeare without using a pretrained language model.

> Run cells from top to bottom in Google Colab. Start with Runtime → Change runtime type → T4 GPU if available. Keep a copy of this notebook in your team's Google Drive/GitHub.

## Team division suggestion
- **Member 1:** tokenizer, dataset, attention, causal-mask tests, hand calculation.
- **Member 2:** Transformer model, training, loss curves, generation.
- **Member 3 (if applicable):** ablations, visualizations, README/report/demo.
Everyone should understand the complete model.

## Important
This notebook implements the architecture directly with PyTorch layers and tensor operations. It does not load a pretrained GPT checkpoint. Record actual results from your own run; do not report placeholder measurements.

## 1. Setup and reproducibility

In [1]:
import os, math, time, random, urllib.request
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("PyTorch:", torch.__version__)

Device: cuda
PyTorch: 2.11.0+cu130


## 2. Download and inspect Tiny Shakespeare

In [2]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
data_path = Path("/content/tinyshakespeare.txt")
if not data_path.exists():
    urllib.request.urlretrieve(DATA_URL, data_path)
text = data_path.read_text(encoding="utf-8")
print("Characters:", len(text))
print(text[:500])

Characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


## 3. Character-level tokenizer and train/validation split
Each unique character receives a deterministic integer ID. This simple tokenizer is allowed by the assignment. The target sequence is shifted one character to the right during training.

In [3]:
chars = sorted(list(set(text)))
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}
vocab_size = len(chars)

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

all_ids = torch.tensor(encode(text), dtype=torch.long)
split_idx = int(0.9 * len(all_ids))
train_ids = all_ids[:split_idx]
val_ids = all_ids[split_idx:]
print("Vocabulary size:", vocab_size)
print("Train tokens:", len(train_ids), "Validation tokens:", len(val_ids))

Vocabulary size: 65
Train tokens: 1003854 Validation tokens: 111540


## 4. Batch creation: next-token prediction

In [4]:
context_length = 128
batch_size = 64

def get_batch(split, batch_size=batch_size, block_size=context_length):
    source = train_ids if split == "train" else val_ids
    ix = torch.randint(len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i:i+block_size] for i in ix])
    y = torch.stack([source[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("Input shape:", tuple(xb.shape), "Target shape:", tuple(yb.shape))
print("Example input:", repr(decode(xb[0][:60].tolist())))
print("Example target:", repr(decode(yb[0][:60].tolist())))

Input shape: (64, 128) Target shape: (64, 128)
Example input: ' passage.\n\nSecond Watchman:\nAy, wherefore else guard we his '
Example target: 'passage.\n\nSecond Watchman:\nAy, wherefore else guard we his r'


## 5. Self-attention from scratch
For each token representation, linear projections create queries (Q), keys (K), and values (V). Scores are scaled by \(\sqrt{d_k}\), future positions are masked, softmax converts scores to weights, and the weights combine V.

In [5]:
class SelfAttentionHead(nn.Module):
    def __init__(self, d_model, head_dim, context_length, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(d_model, head_dim, bias=False)
        self.query = nn.Linear(d_model, head_dim, bias=False)
        self.value = nn.Linear(d_model, head_dim, bias=False)
        self.register_buffer("causal_mask", torch.tril(torch.ones(context_length, context_length)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)       # (B,T,head_dim)
        q = self.query(x)     # (B,T,head_dim)
        v = self.value(x)     # (B,T,head_dim)
        scores = q @ k.transpose(-2, -1) / math.sqrt(k.size(-1))  # (B,T,T)
        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        out = weights @ v
        return out, weights

### 5a. Unit checks for causal masking

In [6]:
torch.manual_seed(SEED)
test_head = SelfAttentionHead(d_model=8, head_dim=4, context_length=8, dropout=0.0).to(device)
test_x = torch.randn(2, 5, 8, device=device)
test_out, test_weights = test_head(test_x)
assert test_out.shape == (2, 5, 4)
assert test_weights.shape == (2, 5, 5)
# Every entry strictly above the diagonal must be zero after softmax.
future_weights = torch.triu(test_weights, diagonal=1)
assert torch.allclose(future_weights, torch.zeros_like(future_weights), atol=1e-7)
print("PASS: attention shapes and future-token masking")
print("Output shape:", tuple(test_out.shape), "Attention shape:", tuple(test_weights.shape))

PASS: attention shapes and future-token masking
Output shape: (2, 5, 4) Attention shape: (2, 5, 5)


## 6. Multi-head attention and sinusoidal positional encoding
Each head attends separately. The outputs are concatenated and projected back to `d_model`. Positional encoding adds information about token order.

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, context_length, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        head_dim = d_model // n_heads
        self.heads = nn.ModuleList([
            SelfAttentionHead(d_model, head_dim, context_length, dropout)
            for _ in range(n_heads)
        ])
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, return_attention=False):
        results = [head(x) for head in self.heads]
        out = torch.cat([r[0] for r in results], dim=-1)
        out = self.dropout(self.proj(out))
        if return_attention:
            return out, [r[1] for r in results]
        return out

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) *
                             (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[:pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

### 6a. Shape trace

In [8]:
shape_test = torch.randn(2, 12, 32, device=device)
mha_test = MultiHeadAttention(d_model=32, n_heads=4, context_length=16).to(device)
mha_out, attn_maps = mha_test(shape_test, return_attention=True)
print("Input:", tuple(shape_test.shape))
print("Each head attention map:", tuple(attn_maps[0].shape))
print("Concatenated/projected output:", tuple(mha_out.shape))
assert mha_out.shape == shape_test.shape

Input: (2, 12, 32)
Each head attention map: (2, 12, 12)
Concatenated/projected output: (2, 12, 32)


## 7. Transformer block and MiniGPT
A block uses pre-layer normalization, causal multi-head attention, residual connections, and a feed-forward MLP. The language-model head maps hidden states to vocabulary logits.

In [9]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, context_length, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, context_length, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.GELU(),
            nn.Linear(4*d_model, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=3,
                 context_length=128, dropout=0.1):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position = SinusoidalPositionalEncoding(d_model, context_length)
        self.blocks = nn.Sequential(*[
            TransformerBlock(d_model, n_heads, context_length, dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.context_length:
            idx = idx[:, -self.context_length:]
            T = idx.size(1)
            if targets is not None:
                targets = targets[:, -T:]
        x = self.token_embedding(idx)
        x = self.position(x)
        x = self.blocks(x)
        logits = self.lm_head(self.final_ln(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200, temperature=0.8, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_length:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            if top_k is not None:
                values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < values[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx

## 8. Instantiate model and run shape/parameter checks

In [10]:
config = {
    "context_length": 128,
    "d_model": 128,
    "n_heads": 4,
    "n_layers": 3,
    "dropout": 0.1,
    "learning_rate": 3e-4,
    "batch_size": 64,
    "max_steps": 1200,
    "eval_interval": 100,
    "eval_batches": 20
}
model = MiniGPT(vocab_size, d_model=config["d_model"], n_heads=config["n_heads"],
                n_layers=config["n_layers"], context_length=config["context_length"],
                dropout=config["dropout"]).to(device)
num_params = sum(p.numel() for p in model.parameters())
print("Parameter count:", f"{num_params:,}")
logits, loss = model(xb, yb)
print("Logits shape:", tuple(logits.shape), "Initial loss:", round(loss.item(), 4))
assert logits.shape == (batch_size, context_length, vocab_size)

Parameter count: 610,625
Logits shape: (64, 128, 65) Initial loss: 4.3678


### 8a. Complete model unit checks
These checks cover tokenizer round-trip, shifted targets, Transformer shapes, finite loss, generation length, and causal attention behavior before long training.


In [11]:
cpu_rng_state = torch.get_rng_state()
cuda_rng_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None

try:
    # Tokenizer round-trip
    round_trip_text = "ROMEO:\n"
    assert decode(encode(round_trip_text)) == round_trip_text

    # Shifted-target correctness
    test_x, test_y = get_batch("train", batch_size=4, block_size=16)
    assert torch.equal(test_x[:, 1:], test_y[:, :-1])

    # Transformer block shape
    test_block = TransformerBlock(
        d_model=32, n_heads=4, context_length=16, dropout=0.0
    ).to(device)
    test_hidden = torch.randn(4, 16, 32, device=device)
    test_block_out = test_block(test_hidden)
    assert test_block_out.shape == test_hidden.shape

    # Full model shape + finite loss
    test_model = MiniGPT(
        vocab_size, d_model=32, n_heads=4, n_layers=2,
        context_length=16, dropout=0.0
    ).to(device)
    test_logits, test_loss = test_model(test_x, test_y)
    assert test_logits.shape == (4, 16, vocab_size)
    assert torch.isfinite(test_loss)

    # Generation length
    test_prompt = torch.tensor([encode("ROMEO:")], dtype=torch.long, device=device)
    test_generated = test_model.generate(test_prompt, max_new_tokens=5, temperature=1.0)
    assert test_generated.shape[1] == test_prompt.shape[1] + 5

    # Future attention weights must be zero
    probe_hidden = torch.randn(2, 10, 32, device=device)
    test_model.eval()
    with torch.no_grad():
        probe_emb = test_model.position(test_model.token_embedding(
            torch.randint(0, vocab_size, (2, 10), device=device)
        ))
        _, probe_maps = test_model.blocks[0].attn(
            test_model.blocks[0].ln1(probe_emb), return_attention=True
        )
    for head_map in probe_maps:
        future = torch.triu(head_map, diagonal=1)
        assert torch.allclose(future, torch.zeros_like(future), atol=1e-7)

    print("PASS: tokenizer, shifted targets, block shapes, loss, generation, and causal attention")
finally:
    del test_block, test_model
    torch.set_rng_state(cpu_rng_state)
    if torch.cuda.is_available():
        torch.cuda.set_rng_state_all(cuda_rng_state)


PASS: tokenizer, shifted targets, block shapes, loss, generation, and causal attention


### 8b. Tiny-overfit debugging gate
Before the full training run, train a very small model on one repeated batch. The test passes only if its loss falls substantially, showing that gradients, target shifting, and optimization can actually learn.


In [12]:
cpu_rng_state = torch.get_rng_state()
cuda_rng_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None

try:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    tiny_context = 32
    tiny_batch_size = 8
    one_x, one_y = get_batch("train", batch_size=1, block_size=tiny_context)
    tiny_x = one_x.repeat(tiny_batch_size, 1)
    tiny_y = one_y.repeat(tiny_batch_size, 1)

    tiny_model = MiniGPT(
        vocab_size,
        d_model=32,
        n_heads=4,
        n_layers=2,
        context_length=tiny_context,
        dropout=0.0
    ).to(device)
    tiny_optimizer = torch.optim.AdamW(tiny_model.parameters(), lr=3e-3)

    tiny_model.train()
    with torch.no_grad():
        _, initial_tiny_loss = tiny_model(tiny_x, tiny_y)
        initial_tiny_loss = initial_tiny_loss.item()

    tiny_loss_history = []
    for tiny_step in range(200):
        _, tiny_loss = tiny_model(tiny_x, tiny_y)
        tiny_optimizer.zero_grad(set_to_none=True)
        tiny_loss.backward()
        tiny_optimizer.step()
        tiny_loss_history.append(tiny_loss.item())

    tiny_model.eval()
    with torch.no_grad():
        _, final_tiny_loss = tiny_model(tiny_x, tiny_y)
        final_tiny_loss = final_tiny_loss.item()

    tiny_overfit_pass = final_tiny_loss < initial_tiny_loss * 0.5
    print(f"Initial tiny-batch loss: {initial_tiny_loss:.4f}")
    print(f"Final tiny-batch loss:   {final_tiny_loss:.4f}")
    print("PASS: tiny overfit gate" if tiny_overfit_pass else "FAIL: tiny overfit gate")
    assert tiny_overfit_pass, "Tiny repeated batch did not overfit enough. Debug before full training."
finally:
    if "tiny_model" in globals():
        del tiny_model
    if "tiny_optimizer" in globals():
        del tiny_optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    torch.set_rng_state(cpu_rng_state)
    if torch.cuda.is_available():
        torch.cuda.set_rng_state_all(cuda_rng_state)


Initial tiny-batch loss: 4.3213
Final tiny-batch loss:   0.0066
PASS: tiny overfit gate


## 9. Training loop with train/validation loss
Run the baseline first. If Colab disconnects or training is too slow, reduce `max_steps` and document the actual value used. A falling training loss is expected, but validation loss and generated samples must also be inspected.

In [13]:
@torch.no_grad()
def estimate_loss(model, eval_batches=20):
    model.eval()
    results = {}
    for split in ("train", "val"):
        losses = []
        for _ in range(eval_batches):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses.append(loss.item())
        results[split] = float(np.mean(losses))
    model.train()
    return results

optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])
train_history, val_history, step_history, lr_history = [], [], [], []
start_time = time.time()
model.train()

for step in range(config["max_steps"] + 1):
    if step % config["eval_interval"] == 0:
        metrics = estimate_loss(model, config["eval_batches"])
        current_lr = optimizer.param_groups[0]["lr"]
        step_history.append(step)
        train_history.append(metrics["train"])
        val_history.append(metrics["val"])
        lr_history.append(current_lr)
        elapsed = time.time() - start_time
        print(
            f"step {step:4d} | train {metrics['train']:.4f} | val {metrics['val']:.4f} "
            f"| lr {current_lr:.2e} | elapsed {elapsed/60:.1f} min"
        )
    if step == config["max_steps"]:
        break
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

training_seconds = time.time() - start_time
print(f"Training time: {training_seconds/60:.2f} minutes")


step    0 | train 4.3647 | val 4.3429 | lr 3.00e-04 | elapsed 0.0 min
step  100 | train 2.5624 | val 2.5684 | lr 3.00e-04 | elapsed 0.1 min
step  200 | train 2.4471 | val 2.4519 | lr 3.00e-04 | elapsed 0.1 min
step  300 | train 2.3332 | val 2.3484 | lr 3.00e-04 | elapsed 0.2 min
step  400 | train 2.2333 | val 2.2537 | lr 3.00e-04 | elapsed 0.2 min
step  500 | train 2.1592 | val 2.1935 | lr 3.00e-04 | elapsed 0.3 min
step  600 | train 2.0921 | val 2.1333 | lr 3.00e-04 | elapsed 0.3 min
step  700 | train 2.0271 | val 2.0815 | lr 3.00e-04 | elapsed 0.4 min
step  800 | train 1.9792 | val 2.0409 | lr 3.00e-04 | elapsed 0.4 min
step  900 | train 1.9232 | val 2.0027 | lr 3.00e-04 | elapsed 0.5 min
step 1000 | train 1.8791 | val 1.9803 | lr 3.00e-04 | elapsed 0.5 min
step 1100 | train 1.8466 | val 1.9546 | lr 3.00e-04 | elapsed 0.6 min
step 1200 | train 1.8035 | val 1.9351 | lr 3.00e-04 | elapsed 0.6 min
Training time: 0.62 minutes


### 9a. Plot the loss curve

In [14]:
plt.figure(figsize=(8, 4))
plt.plot(step_history, train_history, marker="o", label="Training loss")
plt.plot(step_history, val_history, marker="o", label="Validation loss")
plt.xlabel("Training step")
plt.ylabel("Cross-entropy loss")
plt.title("MiniGPT training and validation loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/content/loss_curve.png", dpi=160)
plt.show()

<Figure size 800x400 with 1 Axes>

## 10. Generate text and save a checkpoint

In [15]:
fixed_prompts = ["ROMEO:", "JULIET:", "HAMLET:", "MACBETH:", "KING RICHARD:"]
fixed_generations = {}

model.eval()
for i, prompt in enumerate(fixed_prompts):
    assert all(ch in stoi for ch in prompt), f"Prompt contains a character outside the tokenizer vocabulary: {prompt}"
    torch.manual_seed(SEED + i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED + i)
    prompt_ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    generated_ids = model.generate(
        prompt_ids,
        max_new_tokens=250,
        temperature=0.8,
        top_k=40
    )
    generated_text = decode(generated_ids[0].tolist())
    fixed_generations[prompt] = generated_text
    print("\n" + "=" * 80)
    print("PROMPT:", prompt)
    print(generated_text)

sample = fixed_generations[fixed_prompts[0]]

os.makedirs("/content/checkpoints", exist_ok=True)
checkpoint_path = "/content/checkpoints/minigpt.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "config": config,
    "vocab_size": vocab_size,
    "chars": chars,
    "parameter_count": num_params
}, checkpoint_path)
print("\nSaved checkpoint:", checkpoint_path)

with open("/content/generated_sample.txt", "w", encoding="utf-8") as f:
    f.write(sample)

with open("/content/five_fixed_generations.txt", "w", encoding="utf-8") as f:
    for prompt, generated_text in fixed_generations.items():
        f.write(f"PROMPT: {prompt}\n")
        f.write(generated_text)
        f.write("\n\n" + "=" * 80 + "\n\n")



PROMPT: ROMEO:
ROMEO:
Fit that that denter the me distand he beteck.

GLOUCESTER:
And that will come surt good stater.

LEONTH:
God it didlee man, your hast think.

CADY:

Flair have me mad, not should die to--down

Exicens in marrre, wors theee? word mett 'elm;
Onf whis

PROMPT: JULIET:
JULIET:
Which knot the bet, lies Mery as now than is as our
the coused of my to knoward death, thy sends.

Sirde, for to me of peictied well of way this wealll of
Aglive fir it hene her wind, as
The congle ling at's fellep, netrs from es,
The shallf our to 

PROMPT: HAMLET:
HAMLET:
Loke the my so of dist, is ut all is bount.
And live live thou so's not this beear
havel is is me of love losemend the thanks of tend in be and,
The comen of by his haddd and they be for tat,
Her of manle of Hell if me,
Out of mastinerd: whe or baba

PROMPT: MACBETH:
MACBETH:
Sell't the lead sord ape; by ear ase undo efir,
Thanks each it persed hole the seass, you land.

DUKE VINCENTIO:
Have bear, my for mame my dome of dig

### 10a. Final quantitative evaluation
Evaluate the trained model on fresh train/validation batches and report cross-entropy plus validation perplexity.


In [16]:
final_metrics = estimate_loss(model, eval_batches=50)
final_train_loss = final_metrics["train"]
final_val_loss = final_metrics["val"]
final_val_perplexity = math.exp(min(final_val_loss, 20))

print(f"Final train loss: {final_train_loss:.4f}")
print(f"Final validation loss: {final_val_loss:.4f}")
print(f"Validation perplexity: {final_val_perplexity:.4f}")
print(f"Parameter count: {num_params:,}")
print(f"Training time: {training_seconds/60:.2f} minutes")
print("Device:", device)
print("Learning rate:", optimizer.param_groups[0]["lr"])


Final train loss: 1.8020
Final validation loss: 1.9275
Validation perplexity: 6.8726
Parameter count: 610,625
Training time: 0.62 minutes
Device: cuda
Learning rate: 0.0003


### 10b. Checkpoint reload test
Load the saved weights into a brand-new model object and verify that it produces exactly the same seeded generation as the trained model.


In [17]:
try:
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location=device)

ckpt_config = checkpoint["config"]
reloaded_model = MiniGPT(
    checkpoint["vocab_size"],
    d_model=ckpt_config["d_model"],
    n_heads=ckpt_config["n_heads"],
    n_layers=ckpt_config["n_layers"],
    context_length=ckpt_config["context_length"],
    dropout=ckpt_config["dropout"]
).to(device)
reloaded_model.load_state_dict(checkpoint["model_state_dict"])
reloaded_model.eval()

for original_param, loaded_param in zip(model.parameters(), reloaded_model.parameters()):
    assert torch.equal(original_param.detach(), loaded_param.detach())

reload_prompt = torch.tensor([encode("ROMEO:")], dtype=torch.long, device=device)

torch.manual_seed(2026)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(2026)
original_reload_check = model.generate(
    reload_prompt.clone(), max_new_tokens=40, temperature=0.8, top_k=40
)

torch.manual_seed(2026)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(2026)
loaded_reload_check = reloaded_model.generate(
    reload_prompt.clone(), max_new_tokens=40, temperature=0.8, top_k=40
)

checkpoint_reload_pass = torch.equal(original_reload_check, loaded_reload_check)
assert checkpoint_reload_pass
print("PASS: checkpoint reload reproduces the same seeded generation")


PASS: checkpoint reload reproduces the same seeded generation


### 10c. End-to-end causal correctness test
This is stronger than only inspecting the mask. We alter future tokens and verify that logits at earlier positions do not change.


In [18]:
model.eval()
torch.manual_seed(1234)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1234)

causal_probe_a = torch.randint(0, vocab_size, (1, 24), device=device)
causal_probe_b = causal_probe_a.clone()
cut_position = 10
causal_probe_b[:, cut_position + 1:] = torch.randint(
    0, vocab_size, causal_probe_b[:, cut_position + 1:].shape, device=device
)

with torch.no_grad():
    logits_a, _ = model(causal_probe_a)
    logits_b, _ = model(causal_probe_b)

past_difference = (
    logits_a[:, :cut_position + 1] - logits_b[:, :cut_position + 1]
).abs().max().item()

causal_invariance_pass = past_difference < 1e-5
print("Maximum earlier-logit difference after changing only future tokens:", past_difference)
assert causal_invariance_pass, "Future tokens influenced earlier positions."
print("PASS: end-to-end causal invariance")


Maximum earlier-logit difference after changing only future tokens: 0.0
PASS: end-to-end causal invariance


## 11. Visualize positional encoding and attention
These figures can be used as report evidence. Include notebook/terminal context in screenshots so they clearly come from your implementation.

In [19]:
pe = model.position.pe[0].detach().cpu().numpy()
plt.figure(figsize=(9, 4))
plt.imshow(pe[:64].T, aspect="auto", origin="lower")
plt.xlabel("Token position")
plt.ylabel("Embedding dimension")
plt.title("Sinusoidal positional encoding (first 64 positions)")
plt.colorbar()
plt.tight_layout()
plt.savefig("/content/positional_encoding.png", dpi=160)
plt.show()

# Inspect attention weights in the first head of the first block.
model.eval()
sample_x, _ = get_batch("val", batch_size=1, block_size=32)
with torch.no_grad():
    embedded = model.position(model.token_embedding(sample_x))
    _, attention_maps = model.blocks[0].attn(model.blocks[0].ln1(embedded), return_attention=True)
attention = attention_maps[0][0].detach().cpu().numpy()
plt.figure(figsize=(5, 5))
plt.imshow(attention, origin="lower", aspect="auto")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.title("Trained model: first block, first head")
plt.colorbar()
plt.tight_layout()
plt.savefig("/content/attention_heatmap.png", dpi=160)
plt.show()

<Figure size 900x400 with 2 Axes>

<Figure size 500x500 with 2 Axes>

## 12. Controlled ablations
The assignment asks for at least three experiments. This cell compares depth, number of heads, and context length, changing one factor at a time. To save time, each run is intentionally short; for your report, clearly state the number of steps and treat the measurements as preliminary. Use the same dataset split and seed. Do not compare runs with different step counts as if they were perfectly controlled.

In [20]:
def _current_rss_mb():
    try:
        with open("/proc/self/statm", "r") as f:
            rss_pages = int(f.read().split()[1])
        return rss_pages * os.sysconf("SC_PAGE_SIZE") / (1024 ** 2)
    except Exception:
        return float("nan")

def run_controlled_experiment(d_model=64, n_heads=4, n_layers=4, context_length=128,
                              steps=150, batch=16, lr=3e-4):
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    exp_model = MiniGPT(
        vocab_size,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        context_length=context_length,
        dropout=0.0
    ).to(device)
    opt = torch.optim.AdamW(exp_model.parameters(), lr=lr)

    start_rss = _current_rss_mb()
    peak_rss = start_rss
    t0 = time.time()
    exp_model.train()

    for step in range(steps):
        x, y = get_batch("train", batch_size=batch, block_size=context_length)
        _, exp_loss = exp_model(x, y)
        opt.zero_grad(set_to_none=True)
        exp_loss.backward()
        torch.nn.utils.clip_grad_norm_(exp_model.parameters(), 1.0)
        opt.step()
        if device == "cpu" and step % 10 == 0:
            peak_rss = max(peak_rss, _current_rss_mb())

    elapsed = time.time() - t0
    tokens_seen = steps * batch * context_length
    throughput = tokens_seen / max(elapsed, 1e-9)

    exp_model.eval()
    val_losses = []
    with torch.no_grad():
        for _ in range(10):
            x, y = get_batch("val", batch_size=batch, block_size=context_length)
            _, exp_loss = exp_model(x, y)
            val_losses.append(exp_loss.item())

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    probe = torch.tensor([encode("ROMEO:")], dtype=torch.long, device=device)
    probe_ids = exp_model.generate(probe, max_new_tokens=120, temperature=0.8, top_k=40)
    generation_sample = decode(probe_ids[0].tolist())

    if device == "cuda":
        peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        peak_memory_mb = max(0.0, peak_rss - start_rss)

    result = {
        "d_model": d_model,
        "heads": n_heads,
        "layers": n_layers,
        "context": context_length,
        "steps": steps,
        "batch": batch,
        "parameters": sum(p.numel() for p in exp_model.parameters()),
        "validation_loss": float(np.mean(val_losses)),
        "perplexity": float(math.exp(min(np.mean(val_losses), 20))),
        "training_seconds": round(elapsed, 2),
        "tokens_per_second": round(throughput, 1),
        "peak_memory_mb": round(float(peak_memory_mb), 2),
        "generation_sample": generation_sample
    }

    del exp_model, opt
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

# The same baseline configuration is reused across all three experiment families.
# This avoids retraining the identical 4-layer / 4-head / context-128 model three times.
experiment_cache = {}

def cached_experiment(d_model=64, n_heads=4, n_layers=4, context_length=128):
    key = (d_model, n_heads, n_layers, context_length)
    if key not in experiment_cache:
        experiment_cache[key] = run_controlled_experiment(
            d_model=d_model,
            n_heads=n_heads,
            n_layers=n_layers,
            context_length=context_length
        )
    return dict(experiment_cache[key])

ablation_rows = []

for depth in (2, 4, 6):
    row = cached_experiment(n_layers=depth, n_heads=4, context_length=128)
    row.update({"experiment_family": "depth", "setting": depth})
    ablation_rows.append(row)

for heads in (2, 4, 8):
    row = cached_experiment(n_layers=4, n_heads=heads, context_length=128)
    row.update({"experiment_family": "heads", "setting": heads})
    ablation_rows.append(row)

for ctx in (64, 128, 256):
    row = cached_experiment(n_layers=4, n_heads=4, context_length=ctx)
    row.update({"experiment_family": "context_length", "setting": ctx})
    ablation_rows.append(row)

import pandas as pd
ablation_df = pd.DataFrame(ablation_rows)
display_columns = [
    "experiment_family", "setting", "parameters", "validation_loss", "perplexity",
    "training_seconds", "tokens_per_second", "peak_memory_mb"
]
display(ablation_df[display_columns])

print("\nGeneration samples for the head-count experiment:")
for _, row in ablation_df[ablation_df["experiment_family"] == "heads"].iterrows():
    print("\nHEADS =", row["setting"])
    print(row["generation_sample"])

ablation_df.to_csv("/content/ablation_results.csv", index=False)


,experiment_family,setting,parameters,validation_loss,perplexity,training_seconds,tokens_per_second,peak_memory_mb
0,depth,2,108097,2.769690,15.953689,1.99,154034.6,381.24
1,depth,4,207681,2.700240,14.883310,3.61,85067.4,407.07
2,depth,6,307265,2.637360,13.976260,5.98,51350.4,432.98
3,heads,2,207681,2.695830,14.817817,2.39,128599.7,398.45
4,heads,4,207681,2.700240,14.883310,3.61,85067.4,407.07
5,heads,8,207681,2.688495,14.709516,6.96,44141.2,424.32
6,context_length,64,207681,2.710954,15.043615,4.22,36384.5,376.43
7,context_length,128,207681,2.700240,14.883310,3.61,85067.4,407.07
8,context_length,256,207681,2.710847,15.042017,4.26,144167.5,495.31



Generation samples for the head-count experiment:

HEADS = 2
ROMEO:

3u aeafnte thit te othuared tharnd he berecs tuusenthimEPand be lapfwisers we haroreanl
TAnelu.
GI ouy inde e y d tlat

HEADS = 4
ROMEO:

3u aeafnte thit te othuared tharnd he berecs tuuse thimEPanASAxelapfwisers we haroreanotsAEN:u.
GI ouy inde e y d that

HEADS = 8
ROMEO:

3u asafnte thit te otheared tharnd he berecs turountheXgo- wishelapfwisers we haroreanl
TAnelu.
GI o y inde e y d tlat


## 13. Hand-calculated attention example (report artifact)
Use the matrices below as a transparent numeric example. These are illustrative hand calculations, separate from learned model weights. Compute each step in your report and compare the numbers with the printed values.

We use three tokens and \(d_k=d_v=2\). Since Q=K, the scores are easy to calculate.

In [21]:
Q = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
K = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
V = torch.tensor([[1., 0.], [0., 2.], [3., 1.]])
raw_scores = Q @ K.T
scaled_scores = raw_scores / math.sqrt(2)
weights_unmasked = F.softmax(scaled_scores, dim=-1)
output_unmasked = weights_unmasked @ V
causal = torch.tril(torch.ones(3,3))
masked_scores = scaled_scores.masked_fill(causal == 0, float("-inf"))
weights_masked = F.softmax(masked_scores, dim=-1)
output_masked = weights_masked @ V

print("QK^T =\n", raw_scores)
print("Scaled scores =\n", scaled_scores)
print("Unmasked softmax weights =\n", weights_unmasked)
print("Unmasked output =\n", output_unmasked)
print("Causal masked scores (-inf means blocked) =\n", masked_scores)
print("Masked weights =\n", weights_masked)
print("Masked output =\n", output_masked)

QK^T =
 tensor([[1., 0., 1.],
        [0., 1., 1.],
        [1., 1., 2.]])
Scaled scores =
 tensor([[0.7071, 0.0000, 0.7071],
        [0.0000, 0.7071, 0.7071],
        [0.7071, 0.7071, 1.4142]])
Unmasked softmax weights =
 tensor([[0.4011, 0.1978, 0.4011],
        [0.1978, 0.4011, 0.4011],
        [0.2483, 0.2483, 0.5035]])
Unmasked output =
 tensor([[1.6044, 0.7967],
        [1.4011, 1.2033],
        [1.7587, 1.0000]])
Causal masked scores (-inf means blocked) =
 tensor([[0.7071,   -inf,   -inf],
        [0.0000, 0.7071,   -inf],
        [0.7071, 0.7071, 1.4142]])
Masked weights =
 tensor([[1.0000, 0.0000, 0.0000],
        [0.3302, 0.6698, 0.0000],
        [0.2483, 0.2483, 0.5035]])
Masked output =
 tensor([[1.0000, 0.0000],
        [0.3302, 1.3395],
        [1.7587, 1.0000]])


### Explanation to include in the report
- **QKᵀ:** measures compatibility between each query token and each key token.
- **Scaling:** dividing by √dₖ keeps dot products from becoming excessively large as dimensions grow.
- **Softmax:** turns each row into attention weights that sum to 1.
- **Weighted values:** attention uses those weights to combine information from V.
- **Causal mask:** positions cannot attend to future tokens. Token 1 can see only token 1; token 2 can see tokens 1–2; token 3 can see tokens 1–3. This prevents information leakage during next-token prediction.

## 14. BERT MLM vs GPT causal LM
| Property | BERT-style masked language modeling | GPT-style causal language modeling |
|---|---|---|
| Attention | Bidirectional context is allowed | Future positions are blocked |
| Training target | Predict selected masked tokens | Predict the next token |
| Typical use | Understanding/representation learning | Autoregressive text generation |
| Mask | No causal triangular restriction for ordinary encoder attention | Lower-triangular causal mask |

This assignment requires explaining the difference; it does not require implementing a full BERT model.

### 14a. Short bidirectional-vs-causal attention experiment
A full BERT implementation is not needed. This controlled check uses the same Q, K, and V and changes only the attention constraint so the effect of bidirectional versus causal masking is directly visible.


In [22]:
def scaled_attention_demo(Q, K, V, causal=False):
    scores = Q @ K.T / math.sqrt(Q.size(-1))
    if causal:
        mask = torch.tril(torch.ones(scores.size(-2), scores.size(-1), dtype=torch.bool))
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

demo_Q = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
demo_K = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
demo_V_a = torch.tensor([[1., 0.], [0., 2.], [3., 1.]])
demo_V_b = demo_V_a.clone()
demo_V_b[2] = torch.tensor([30., -10.])

bidirectional_a, _ = scaled_attention_demo(demo_Q, demo_K, demo_V_a, causal=False)
bidirectional_b, _ = scaled_attention_demo(demo_Q, demo_K, demo_V_b, causal=False)
causal_a, _ = scaled_attention_demo(demo_Q, demo_K, demo_V_a, causal=True)
causal_b, _ = scaled_attention_demo(demo_Q, demo_K, demo_V_b, causal=True)

bidirectional_change = (bidirectional_a[0] - bidirectional_b[0]).abs().max().item()
causal_change = (causal_a[0] - causal_b[0]).abs().max().item()

print("First-position change when a future value changes")
print("Bidirectional attention:", bidirectional_change)
print("Causal attention:       ", causal_change)

assert bidirectional_change > 1e-6
assert causal_change < 1e-7
print("PASS: bidirectional attention can use future information; causal attention cannot")

causal_mask_figure = torch.tril(torch.ones(12, 12))
print("\nCausal mask (1 = visible, 0 = blocked):\n", causal_mask_figure.int())
plt.figure(figsize=(5, 5))
plt.imshow(causal_mask_figure.numpy(), origin="upper", aspect="equal")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.title("Causal attention mask")
plt.tight_layout()
plt.savefig("/content/causal_mask.png", dpi=160)
plt.show()


First-position change when a future value changes
Bidirectional attention: 10.830026626586914
Causal attention:        0.0
PASS: bidirectional attention can use future information; causal attention cannot

Causal mask (1 = visible, 0 = blocked):
 tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], dtype=torch.int32)


<Figure size 500x500 with 1 Axes>

## 14b. Final model-and-testing acceptance checks
This cell does not cover README, API/deployment, report writing, screenshots, or presentation. It only verifies the model/training/testing work completed in this notebook.


In [23]:
model_testing_checks = {
    "tiny_overfit_gate": bool(tiny_overfit_pass),
    "five_fixed_generations": len(fixed_generations) >= 5,
    "checkpoint_reload": bool(checkpoint_reload_pass),
    "end_to_end_causal_invariance": bool(causal_invariance_pass),
    "finite_final_validation_loss": bool(np.isfinite(final_val_loss)),
    "finite_validation_perplexity": bool(np.isfinite(final_val_perplexity)),
    "depth_experiment_2_4_6": set(
        ablation_df.loc[ablation_df["experiment_family"] == "depth", "setting"].astype(int)
    ) == {2, 4, 6},
    "head_experiment_2_4_8": set(
        ablation_df.loc[ablation_df["experiment_family"] == "heads", "setting"].astype(int)
    ) == {2, 4, 8},
    "context_experiment_64_128_256": set(
        ablation_df.loc[ablation_df["experiment_family"] == "context_length", "setting"].astype(int)
    ) == {64, 128, 256},
    "ablation_throughput_measured": bool(
        np.isfinite(ablation_df["tokens_per_second"]).all()
    ),
    "ablation_memory_measured": bool(
        np.isfinite(ablation_df["peak_memory_mb"]).all()
    ),
    "learning_rate_logged": len(lr_history) == len(step_history) and len(lr_history) > 0,
}

for check_name, passed in model_testing_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} - {check_name}")

assert all(model_testing_checks.values()), "One or more model/testing requirements still failed."
print("\nALL MODEL + TESTING CHECKS PASSED")


PASS - tiny_overfit_gate
PASS - five_fixed_generations
PASS - checkpoint_reload
PASS - end_to_end_causal_invariance
PASS - finite_final_validation_loss
PASS - finite_validation_perplexity
PASS - depth_experiment_2_4_6
PASS - head_experiment_2_4_8
PASS - context_experiment_64_128_256
PASS - ablation_throughput_measured
PASS - ablation_memory_measured
PASS - learning_rate_logged

ALL MODEL + TESTING CHECKS PASSED


## 15. Export artifacts
Download these from Colab's Files pane or copy them to Google Drive. Do not commit large files unless your instructor asks. A checkpoint stored only in the temporary Colab filesystem will disappear when the runtime resets.

In [24]:
print("Artifacts:")
for p in ["/content/loss_curve.png", "/content/positional_encoding.png",
          "/content/attention_heatmap.png", "/content/ablation_results.csv",
          "/content/generated_sample.txt", "/content/checkpoints/minigpt.pt"]:
    print(p, "exists:", os.path.exists(p))

Artifacts:
/content/loss_curve.png exists: True
/content/positional_encoding.png exists: True
/content/attention_heatmap.png exists: True
/content/ablation_results.csv exists: True
/content/generated_sample.txt exists: True
/content/checkpoints/minigpt.pt exists: True


In [25]:
import os
import zipfile

files_to_save=[
    "/content/checkpoints/minigpt.pt",
    "/content/loss_curve.png",
    "/content/positional_encoding.png",
    "/content/attention_heatmap.png",
    "/content/ablation_results.csv",
    "/content/five_fixed_generations.txt"
]

with zipfile.ZipFile("/content/MiniGPT_Final_Results.zip","w") as z:
    for f in files_to_save:
        if os.path.exists(f):
            z.write(f,os.path.basename(f))
            print("Saved",f)
        else:
            print("Not found",f)

print("Done")

Saved /content/checkpoints/minigpt.pt
Saved /content/loss_curve.png
Saved /content/positional_encoding.png
Saved /content/attention_heatmap.png
Saved /content/ablation_results.csv
Saved /content/five_fixed_generations.txt
Done


In [26]:
import os
import zipfile
from google.colab import files

wanted_files=[
    "minigpt.pt",
    "loss_curve.png",
    "positional_encoding.png",
    "attention_heatmap.png",
    "ablation_results.csv",
    "five_fixed_generations.txt"
]

found=[]

for root,dirs,filenames in os.walk("/content"):
    for name in filenames:
        if name in wanted_files:
            path=os.path.join(root,name)
            found.append(path)

zip_path="/content/MiniGPT_FINAL_RESULTS.zip"

with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for path in found:
        z.write(path,arcname=os.path.basename(path))

print("FILES FOUND AND SAVED")
for path in found:
    print("✓",path)

missing=[name for name in wanted_files
         if name not in [os.path.basename(x) for x in found]]

if missing:
    print("\nFILES NOT FOUND")
    for name in missing:
        print("✗",name)

print("\nDownloading MiniGPT_FINAL_RESULTS.zip...")
files.download(zip_path)

FILES FOUND AND SAVED
✓ /content/loss_curve.png
✓ /content/positional_encoding.png
✓ /content/attention_heatmap.png
✓ /content/five_fixed_generations.txt
✓ /content/ablation_results.csv
✓ /content/checkpoints/minigpt.pt



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>